In [2]:
import numpy as np

After integrating the dynamics for one time step, we have:

* x: Neuronal state before the step
* xnew: Neuronal state after the step
* rnew: Updated output or filtered spike variable
* nspike: Spike counter for this time step
* nqif: Number of QIF neurons

The functions purpose is to determine whether each neuron crossed its spike threshold during the transition $x \rightarrow x_{new}$

**LIF spike detection**
```ispike_lif = np.where(x[nqif:] < vt) and np.where(xnew[nqif:] > vt)``` 

A LIF neuron spikes when it was below threshold before the step, and it is above threshold after the step. For neuron *i*, that means $x_i < v_t$ and $x_{i}^{new} > v_t$.

This is the previous version:

In [ ]:
def detect(x,xnew,rnew,nspike,nqif, b, vt, vrest):

    # LIF spike detection
    ispike_lif=np.where(x[nqif:]<vt) and np.where(xnew[nqif:]>vt)
    ispike_lif=ispike_lif[0]+nqif
    if(len(ispike_lif)>0):
        rnew[ispike_lif[:]] = rnew[ispike_lif[:]] + b
        xnew[ispike_lif[:]] = vrest
        nspike[ispike_lif[:]] = nspike[ispike_lif[:]] + 1

    # QIF spike detection
    dpi=np.mod(np.pi - np.mod(x,2*np.pi),2*np.pi)  # distance to pi
    ispike_qif=np.where((xnew[:nqif]-x[:nqif])>0) and np.where((xnew[:nqif]-x[:nqif]-dpi[:nqif])>0)
    if(len(ispike_qif)>0):
        rnew[ispike_qif[:]] = rnew[ispike_qif[:]] + b
        nspike[ispike_qif[:]] = nspike[ispike_qif[:]] + 1
    return xnew,rnew,nspike

First line: ```ispike_lif=np.where(x[nqif:]<vt) and np.where(xnew[nqif:]>vt)```

Let's say we have this state:

```bash

np.where(x[nqif:]<vt)
> (array([], dtype=int64),)  # no neurons were below threshold

np.where(xnew[nqif:]>vt)
> (array([0, 2, 4, 6, 7, 8, 9]),)  # these neurons were above threshold

np.where(x[nqif:]<vt) and np.where(xnew[nqif:]>vt)
> (array([0, 2, 4, 6, 7, 8, 9]),)  # the condition A and B in this case returns neurons meeting only the last condition, even though what we want is to only keep those that meet both

```
    
Second line: ```ispike_lif=ispike_lif[0]+nqif``` 
.. pushes the indices with nqif steps, i.e. if the first 10 neurons are QIF, then the indices returned from previous step needs to start from past these. 

```bash

ispike_lif[0]
> array([0, 1, 3, 4, 5, 8, 9])

print(ispike_lif[0]+nqif)  # add nqif
> [10 11 13 14 15 18 19]
```

If ispike is empty, it stays empty, because NumPy does elementwise addition.

**QIF spike detection**

We think of the neuron's state as moving around a circle:


```bash

                   π
                 SPIKE
                   ●
              ↗         ↘

       π/2 ●                 ● 3π/2

              ↖         ↙
                   ●
                 0 = 2π
```



In [3]:
vt = 0
nqif = 1

x = [-1, -1, -0.1, 0, 1]
x = np.array(x)

print(np.where(x[nqif:] < vt))

(array([0, 1]),)


Python's ```and``` does not combine NumPy arrays element by element. Instead, the expression ```A and B``` means:

* evaluate whether A is true
* if A is true, return B
* otherwise return A

It does not mean 

```
A[0] and B[0]
A[1] and B[1]
...
```

In [ ]:
# An array containing only false will still return True by this notation

truth_value = bool(np.where(np.array([False, False])))

print(truth_value)


True


(array([2]),)

Now an example of why Python's ```and``` doesn't work properly:

In [2]:
import numpy as np

N = 10
pqif = 0.5
nqif = int(pqif * N)   # First 5 are QIF, last 5 are LIF

# States before and after one timestep
x = np.zeros(N)
xnew = np.zeros(N)

# Only fill the LIF neurons (indices 5-9)
x[nqif:] = [-1.0, 0.5, -0.2, -0.5, 0.2]
xnew[nqif:] = [0.3, 0.8, -0.1, 0.2, -0.4]

vt = 0

print("Before threshold:", x[nqif:] < vt)
print("After threshold :", xnew[nqif:] > vt)

print("\nCorrect (elementwise &):")
print((x[nqif:] < vt) & (xnew[nqif:] > vt))

print("\nIndices:")
print(np.where((x[nqif:] < vt) & (xnew[nqif:] > vt))[0])

print("\nCurrent code:")
print(np.where(x[nqif:] < vt) and np.where(xnew[nqif:] > vt))

Before threshold: [ True False  True  True False]
After threshold : [ True  True False  True False]

Correct (elementwise &):
[ True False False  True False]

Indices:
[0 3]

Current code:
(array([0, 1, 3]),)


In [ ]:
def detect(x,xnew,rnew,nspike,nqif, b, vt, vrest):

    # LIF spike detection
   
    ispike_lif=np.where(x[nqif:]<vt) and np.where(xnew[nqif:]>vt) # PROBLEM: 'and' is Pythons logical operator, not NumPy's elementwise AND

    ispike_lif=ispike_lif[0]+nqif
    if(len(ispike_lif)>0):
        rnew[ispike_lif[:]] = rnew[ispike_lif[:]] + b
        xnew[ispike_lif[:]] = vrest
        nspike[ispike_lif[:]] = nspike[ispike_lif[:]] + 1

    # QIF spike detection
    dpi=np.mod(np.pi - np.mod(x,2*np.pi),2*np.pi)  # distance to pi. PROBLEM: Works, but clearer to compute only for QIF neurons x[:nqif]

    ispike_qif=np.where((xnew[:nqif]-x[:nqif])>0) and np.where((xnew[:nqif]-x[:nqif]-dpi[:nqif])>0)  # PROBLEM: Possibly wrong and operator

    if(len(ispike_qif)>0):  #PROBLEM: ispike_qif is still a tuple from np.where, so len(..) is always 1, even if no neurons spike
        rnew[ispike_qif[:]] = rnew[ispike_qif[:]] + b
        nspike[ispike_qif[:]] = nspike[ispike_qif[:]] + 1
    return xnew,rnew,nspike

We might use this code instead:

In [16]:
def detect(x, xnew, rnew, nspike, nqif, b, vt, vrest):

    # LIF spike detection
    ispike_lif = np.where(
        (x[nqif:] < vt) & (xnew[nqif:] > vt)
    )[0] + nqif

    if len(ispike_lif) > 0:
        rnew[ispike_lif] += b
        xnew[ispike_lif] = vrest
        nspike[ispike_lif] += 1

    # QIF spike detection
    delta_x = xnew[:nqif] - x[:nqif]

    dpi = np.mod(
        np.pi - np.mod(x[:nqif], 2 * np.pi),
        2 * np.pi
    )

    ispike_qif = np.where(
        (delta_x > 0) & (delta_x - dpi > 0)
    )[0]

    if len(ispike_qif) > 0:
        rnew[ispike_qif] += b
        nspike[ispike_qif] += 1

    return xnew, rnew, nspike